# Lab 8: Kafka Integration with Iceberg - Solution

This notebook provides the complete solution for Lab 8, demonstrating how to integrate Apache Kafka with Apache Iceberg for real-time data ingestion.

## Setup

First, let's configure Spark with Iceberg and Kafka support.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json
import time
import random
from datetime import datetime

# Create Spark session with Iceberg and Kafka support
spark = SparkSession.builder \
    .appName("Kafka-Iceberg-Solution") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "rest") \
    .config("spark.sql.catalog.iceberg.uri", "http://polaris:8181/api/catalog") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.streaming.checkpointLocation", "s3a://spark-checkpoints/kafka-iceberg") \
    .getOrCreate()

## Part 1: Create Iceberg Tables for Streaming Data

In [ ]:
# Create orders table
spark.sql("""
CREATE TABLE IF NOT EXISTS iceberg.stream.orders (
    order_id BIGINT,
    customer_id INT,
    customer_name STRING,
    customer_email STRING,
    product_id INT,
    product_name STRING,
    quantity INT,
    unit_price DECIMAL(10, 2),
    total_amount DECIMAL(10, 2),
    order_date TIMESTAMP,
    status STRING,
    processed_timestamp TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(order_date))
""")

print("Orders table created successfully")

## Part 2: Simulate Kafka Producer

Since we can't directly run a Kafka producer from this notebook, we'll simulate the data that would come from Kafka.

In [ ]:
# Sample customer and product data
customers = [
    {'id': 1, 'name': 'John Doe', 'email': 'john@example.com'},
    {'id': 2, 'name': 'Jane Smith', 'email': 'jane@example.com'},
    {'id': 3, 'name': 'Bob Johnson', 'email': 'bob@example.com'},
]

products = [
    {'id': 1, 'name': 'Laptop Pro 15"', 'price': 1299.99},
    {'id': 2, 'name': 'Wireless Mouse', 'price': 29.99},
    {'id': 3, 'name': 'Mechanical Keyboard', 'price': 149.99},
]

def generate_order():
    customer = random.choice(customers)
    product = random.choice(products)
    quantity = random.randint(1, 5)
    
    return {
        'order_id': int(time.time() * 1000),
        'customer_id': customer['id'],
        'customer_name': customer['name'],
        'customer_email': customer['email'],
        'product_id': product['id'],
        'product_name': product['name'],
        'quantity': quantity,
        'unit_price': product['price'],
        'total_amount': product['price'] * quantity,
        'order_date': datetime.utcnow().isoformat(),
        'status': random.choice(['pending', 'processing', 'shipped', 'delivered'])
    }

# Generate sample orders
sample_orders = [generate_order() for _ in range(100)]

# Create DataFrame from sample orders
orders_df = spark.createDataFrame(sample_orders)
orders_df.show(5)

## Part 3: Process Streaming Data (Simulated)

In [ ]:
# Process the orders as if they came from Kafka
processed_orders = orders_df.withColumn(
    "processed_timestamp", current_timestamp()
)

# Write to Iceberg
processed_orders.write \
    .format("iceberg") \
    .mode("append") \
    .saveAsTable("iceberg.stream.orders")

print("Orders written to Iceberg successfully")

## Part 4: Real-Time Analytics Queries

In [ ]:
# Real-time order statistics
order_stats = spark.sql("""
SELECT 
    status,
    COUNT(*) as order_count,
    SUM(total_amount) as total_revenue,
    AVG(total_amount) as avg_order_value
FROM iceberg.stream.orders
GROUP BY status
ORDER BY total_revenue DESC
""")

print("Order Statistics:")
order_stats.show()

In [ ]:
# Real-time product popularity
product_popularity = spark.sql("""
SELECT 
    product_id,
    product_name,
    COUNT(*) as order_count,
    SUM(quantity) as total_quantity,
    SUM(total_amount) as total_revenue
FROM iceberg.stream.orders
GROUP BY product_id, product_name
ORDER BY total_revenue DESC
LIMIT 10
""")

print("Product Popularity:")
product_popularity.show()

In [ ]:
# Real-time customer activity
customer_activity = spark.sql("""
SELECT 
    customer_id,
    customer_name,
    customer_email,
    COUNT(*) as order_count,
    SUM(total_amount) as total_spent
FROM iceberg.stream.orders
GROUP BY customer_id, customer_name, customer_email
ORDER BY total_spent DESC
LIMIT 10
""")

print("Customer Activity:")
customer_activity.show()

## Part 5: Data Quality Validation

In [ ]:
# Data quality checks
quality_checks = spark.sql("""
SELECT 
    COUNT(*) as total_records,
    COUNT(DISTINCT order_id) as unique_orders,
    COUNT(DISTINCT customer_id) as unique_customers,
    COUNT(DISTINCT product_id) as unique_products,
    SUM(CASE WHEN quantity > 0 THEN 1 ELSE 0 END) as valid_quantity,
    SUM(CASE WHEN unit_price > 0 THEN 1 ELSE 0 END) as valid_price,
    SUM(CASE WHEN total_amount > 0 THEN 1 ELSE 0 END) as valid_total,
    SUM(CASE WHEN customer_email LIKE '%@%' THEN 1 ELSE 0 END) as valid_email
FROM iceberg.stream.orders
""")

print("Data Quality Checks:")
quality_checks.show()

## Part 6: Streaming Window Aggregations (Simulated)

In [ ]:
# Simulate window aggregations
from pyspark.sql.window import Window

# Add window columns
windowed_orders = orders_df.withColumn(
    "hour_window", date_trunc("hour", col("order_date"))
).withColumn(
    "day_window", date_trunc("day", col("order_date"))
)

# Hourly aggregation
hourly_agg = windowed_orders.groupBy(
    "hour_window", "status"
).agg(
    count("*").alias("order_count"),
    sum("total_amount").alias("total_revenue")
)

print("Hourly Aggregations:")
hourly_agg.orderBy("hour_window").show()

## Part 7: Schema Evolution

In [ ]:
# Add new columns to support schema evolution
spark.sql("""
ALTER TABLE iceberg.stream.orders 
ADD COLUMN discount_code STRING
""")

spark.sql("""
ALTER TABLE iceberg.stream.orders 
ADD COLUMN shipping_method STRING
""")

spark.sql("""
ALTER TABLE iceberg.stream.orders 
ADD COLUMN payment_method STRING
""")

print("Schema evolved successfully")

In [ ]:
# Verify new schema
spark.sql("DESCRIBE iceberg.stream.orders").show()

In [ ]:
# Insert data with new fields
enhanced_orders = orders_df.withColumn(
    "discount_code", lit("NONE")
).withColumn(
    "shipping_method", lit("STANDARD")
).withColumn(
    "payment_method", lit("CREDIT_CARD")
).withColumn(
    "processed_timestamp", current_timestamp()
)

enhanced_orders.write \
    .format("iceberg") \
    .mode("append") \
    .saveAsTable("iceberg.stream.orders")

print("Enhanced orders written successfully")

## Part 8: Monitor Streaming Data

In [ ]:
# Check table snapshots
snapshots = spark.sql("CALL iceberg.system.history('iceberg.stream.orders')")
print("Table Snapshots:")
snapshots.show()

In [ ]:
# Check data freshness
data_freshness = spark.sql("""
SELECT 
    current_timestamp() - MAX(processed_timestamp) as processing_lag_seconds,
    current_timestamp() - MAX(order_date) as data_lag_seconds
FROM iceberg.stream.orders
""")

print("Data Freshness:")
data_freshness.show()

## Cleanup

In [ ]:
# Drop the table
spark.sql("DROP TABLE IF EXISTS iceberg.stream.orders")

print("Cleanup completed successfully")

## Summary

This solution demonstrated:

1. **Iceberg Table Creation**: Created partitioned Iceberg tables for streaming data
2. **Data Ingestion**: Simulated Kafka message ingestion with proper schema
3. **Real-Time Analytics**: Implemented real-time aggregation queries
4. **Data Quality**: Added validation checks for streaming data
5. **Window Aggregations**: Implemented time-based window aggregations
6. **Schema Evolution**: Demonstrated adding new columns without breaking the pipeline
7. **Monitoring**: Added data freshness and snapshot monitoring

In a production environment, you would:
- Use actual Kafka topics instead of simulated data
- Run the streaming query continuously with `awaitTermination()`
- Implement proper error handling and retry logic
- Add monitoring and alerting for the streaming pipeline
- Configure exactly-once processing semantics